# Feature engineering

## 0. Configs

### 0.1 Imports

In [2]:
%load_ext autoreload
%autoreload 2

In [13]:
import pandas as pd
import numpy as np
import warnings

# configurações para importar as funcões do módulo utils
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# funções utils
from src.utils_eda import months, days, faixa_etaria

# configurações de exibição
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

### 0.2 Base de dados

In [14]:
# ler tabela
df1 = pd.read_csv('../data/raw/bank-marketing-data-set/bank-additional-full.csv', sep = ';')

# tratar nomes dos meses e dias da semana
df1['month'] = df1['month'].map(months)
df1['day_of_week'] = df1['day_of_week'].map(days)

# criar coluna de ano
df1['month_num'] = df1['month'].str[:2].astype(int)
virada_ano = df1['month_num'] < df1['month_num'].shift()
df1['year'] = 2008 + virada_ano.cumsum()
df1 = df1.drop(columns='month_num')

# criar coluna de faixa etária
df1['faixa_etaria'] = df1['age'].apply(faixa_etaria)

# substituir . por _ em nomes de colunas
df1.columns = [c.replace('.', '_') for c in df1.columns]

print(f"Tamanho do dataset: {df1.shape}")

df1.head()

Tamanho do dataset: (41188, 23)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y,year,faixa_etaria
0,56,housemaid,married,basic.4y,no,no,no,telephone,05. may,1. mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2008,Entre 50 e 60 anos
1,57,services,married,high.school,unknown,no,no,telephone,05. may,1. mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2008,Entre 50 e 60 anos
2,37,services,married,high.school,no,yes,no,telephone,05. may,1. mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2008,Entre 31 e 40 anos
3,40,admin.,married,basic.6y,no,no,no,telephone,05. may,1. mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2008,Entre 31 e 40 anos
4,56,services,married,high.school,no,no,yes,telephone,05. may,1. mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no,2008,Entre 50 e 60 anos


## 1. Feature engineering

In [ ]:
# inicializar tabela
df_modelo = df1.copy()


# selecionar colunas
df_modelo = df_modelo[[
    # perfil
    'age', 'job', 'marital', 'education', 'faixa_etaria',

    # situação financeira
    'default', 'housing', 'loan', 

    # dados dos contatos
    'contact', 'month', 'day_of_week',

    # histórico de relacionamento
    'previous', 'poutcome',

    # indicadores econômicos
    'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed',

    # target
    'y'
]]

## removidas:
# - campaign; não faz sentido usar a quantidade de contatos porque isso não necessariamente será usado ao fazer o predict
# - duration: não tem como saber qual será a duração da ligação ao fazer o contato
# - pdays: a maioria é 999 (equivalente a null)

# criar variáveis derivadas





In [8]:
colunas = ['emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

dicionario = {
    col: {
        'media': df1[col].mean(),
        'min': df1[col].min(),
        'mediana': df1[col].median(),
        'max': df1[col].max()
    }
    for col in colunas
}

dicionario


{'emp_var_rate': {'media': np.float64(0.08188550063125184),
  'min': np.float64(-3.4),
  'mediana': np.float64(1.1),
  'max': np.float64(1.4)},
 'cons_price_idx': {'media': np.float64(93.57566436826262),
  'min': np.float64(92.201),
  'mediana': np.float64(93.749),
  'max': np.float64(94.767)},
 'cons_conf_idx': {'media': np.float64(-40.50260027192386),
  'min': np.float64(-50.8),
  'mediana': np.float64(-41.8),
  'max': np.float64(-26.9)},
 'euribor3m': {'media': np.float64(3.6212908128581147),
  'min': np.float64(0.634),
  'mediana': np.float64(4.857),
  'max': np.float64(5.045)},
 'nr_employed': {'media': np.float64(5167.035910944935),
  'min': np.float64(4963.6),
  'mediana': np.float64(5191.0),
  'max': np.float64(5228.1)}}